# Notebook 04 — Cải tiến mô hình 8 bước + Threshold Optimization + Đóng gói


Phiên bản tinh gọn: giữ lại đúng 8 kỹ thuật cải tiến liên tiếp mà mỗi bước đều **tăng F1** so với bước trước (không bước nào đi lùi), thể hiện rõ đường cong "lợi ích giảm dần" (diminishing returns).

**Nguyên tắc bất biến dữ liệu** giữ nguyên như Notebook 02/03: không sửa dữ liệu gốc, mọi imputation chỉ nằm trong Pipeline.

## 0. Kết nối Drive & cài thư viện

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/DSS_Project'
    print('Đã kết nối Google Drive thành công!')
except ImportError:
    BASE_DIR = '.'
    print('Không ở trong môi trường Colab, sử dụng đường dẫn local.')

In [ ]:
!pip install -q imbalanced-learn optuna shap xgboost lightgbm pytabkit

In [ ]:
import os
import json
import pickle
import warnings
import inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, RepeatedStratifiedKFold, cross_validate, cross_val_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, StackingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    roc_auc_score, average_precision_score, recall_score, precision_score,
    f1_score, fbeta_score, accuracy_score, confusion_matrix, precision_recall_curve,
    ConfusionMatrixDisplay, classification_report
)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

import xgboost as xgb
import lightgbm as lgb

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import shap

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 150)

RANDOM_STATE = 42
DATA_DIR = os.path.join(BASE_DIR, 'Data_cleaned')
MODEL_DIR = os.path.join(BASE_DIR, '04_models')
os.makedirs(MODEL_DIR, exist_ok=True)
print('Thư mục dữ liệu:', DATA_DIR)
print('Thư mục lưu model:', MODEL_DIR)

## 1. Tái tạo dữ liệu & `df_model` (giống Notebook 02/03 — chỉ đọc, không sửa dữ liệu gốc)

In [ ]:
TABLE_FILES = {
    'account': 'completedacct.csv', 'client': 'completedclient.csv',
    'disposition': 'completeddisposition.csv', 'district': 'completeddistrict.csv',
    'loan': 'completedloan.csv', 'card': 'completedcard.csv',
    'order': 'completedorder.csv', 'trans': 'completedtrans.csv',
}
raw = {name: pd.read_csv(os.path.join(DATA_DIR, f)) for name, f in TABLE_FILES.items()}
ORIG_SHAPES = {name: df.shape for name, df in raw.items()}
ORIG_NULL_COUNTS = {name: int(df.isnull().sum().sum()) for name, df in raw.items()}
for name, df in raw.items():
    print(f'  - {name:12s}: {df.shape[0]:>9,} dòng x {df.shape[1]:>2} cột')

In [ ]:
STATUS_TO_TARGET = {'A': 0, 'C': 0, 'B': 1, 'D': 1}
loan_labeled = raw['loan'].copy()
loan_labeled['target'] = loan_labeled['status'].map(STATUS_TO_TARGET)
loan_labeled['loan_date_parsed'] = pd.to_datetime(loan_labeled['date'], errors='coerce')
loan_labeled = loan_labeled[loan_labeled['target'].notnull()].copy()
loan_labeled['target'] = loan_labeled['target'].astype(int)

account = raw['account'].copy()
account['account_open_date_parsed'] = pd.to_datetime(account['date'], errors='coerce')
district = raw['district'].copy()
district_small = district[['district_id', 'region', 'division']].rename(columns={'district_id': 'account_district_id'})
account_features = account[['account_id', 'district_id', 'frequency', 'account_open_date_parsed']].rename(
    columns={'district_id': 'account_district_id'}
).merge(district_small, on='account_district_id', how='left')

disp = raw['disposition'].copy()
disp['client_id'] = disp['client_id'].astype(str)
client = raw['client'].copy()
client['client_id'] = client['client_id'].astype(str)
n_disponents_df = disp.groupby('account_id').size().reset_index(name='n_disponents')
owner_df = disp[disp['type'] == 'OWNER'][['account_id', 'client_id']].drop_duplicates(subset='account_id')
owner_client = owner_df.merge(client[['client_id', 'sex', 'age', 'district_id']], on='client_id', how='left').rename(
    columns={'sex': 'owner_sex', 'age': 'owner_age', 'district_id': 'owner_district_id'}
)
owner_client = owner_client.merge(
    account[['account_id', 'district_id']].rename(columns={'district_id': 'account_district_id'}),
    on='account_id', how='left'
)
owner_client['is_local_client'] = (owner_client['owner_district_id'] == owner_client['account_district_id']).astype(int)
client_features = owner_client[['account_id', 'owner_sex', 'owner_age', 'is_local_client']]

card = raw['card'].copy()
card['card_date_parsed'] = pd.to_datetime(card['date'], errors='coerce')
card_disp = card.merge(disp[['disp_id', 'account_id']], on='disp_id', how='left')
card_loan = card_disp.merge(loan_labeled[['account_id', 'loan_id', 'loan_date_parsed']], on='account_id', how='inner')
card_loan['has_card_flag'] = (card_loan['card_date_parsed'] < card_loan['loan_date_parsed']).astype(int)
has_card_df = card_loan.groupby('loan_id')['has_card_flag'].max().reset_index().rename(columns={'has_card_flag': 'has_card_before_loan'})

order = raw['order'].copy()
order_features = order.groupby('account_id').agg(
    n_orders=('order_id', 'count'), total_order_amount=('amount', 'sum'), avg_order_amount=('amount', 'mean')
).reset_index()

trans = raw['trans'].copy()
trans['trans_date_parsed'] = pd.to_datetime(trans['date'], errors='coerce')
trans_loan = trans.merge(loan_labeled[['account_id', 'loan_id', 'loan_date_parsed']], on='account_id', how='inner')
filtered = trans_loan[trans_loan['trans_date_parsed'] < trans_loan['loan_date_parsed']].copy()
trans_agg = filtered.groupby('loan_id').agg(
    n_trans_before_loan=('trans_id', 'count'), avg_balance_before_loan=('balance', 'mean'),
    std_balance_before_loan=('balance', 'std'), min_balance_before_loan=('balance', 'min'),
    max_balance_before_loan=('balance', 'max'), avg_amount_before_loan=('amount', 'mean'),
    total_amount_before_loan=('amount', 'sum'),
).reset_index()
last_balance = filtered.sort_values(['loan_id', 'trans_date_parsed']).groupby('loan_id').tail(1)[
    ['loan_id', 'balance']
].rename(columns={'balance': 'last_balance_before_loan'})
trans_features = trans_agg.merge(last_balance, on='loan_id', how='left')

print('Đã tái tạo xong toàn bộ các bảng đặc trưng trung gian.')

In [ ]:
df_model = loan_labeled[['loan_id', 'account_id', 'amount', 'duration', 'payments', 'purpose', 'status', 'target']].copy()
df_model = df_model.merge(account_features, on='account_id', how='left')
df_model = df_model.merge(loan_labeled[['loan_id', 'loan_date_parsed']], on='loan_id', how='left')
df_model['account_age_days'] = (df_model['loan_date_parsed'] - df_model['account_open_date_parsed']).dt.days
df_model = df_model.merge(client_features, on='account_id', how='left')
df_model = df_model.merge(n_disponents_df, on='account_id', how='left')
df_model = df_model.merge(has_card_df, on='loan_id', how='left')
df_model = df_model.merge(order_features, on='account_id', how='left')
df_model = df_model.merge(trans_features, on='loan_id', how='left')

ZERO_STRUCTURAL_COLS = ['n_disponents', 'has_card_before_loan', 'n_orders', 'total_order_amount',
                        'avg_order_amount', 'n_trans_before_loan', 'total_amount_before_loan']
for col in ZERO_STRUCTURAL_COLS:
    df_model[col] = df_model[col].fillna(0)

print('df_model:', df_model.shape)

numeric_features = ['amount', 'duration', 'payments', 'account_age_days', 'owner_age', 'n_disponents',
                     'n_orders', 'total_order_amount', 'avg_order_amount',
                     'n_trans_before_loan', 'avg_balance_before_loan', 'std_balance_before_loan',
                     'min_balance_before_loan', 'max_balance_before_loan', 'last_balance_before_loan',
                     'avg_amount_before_loan', 'total_amount_before_loan']
categorical_features = ['purpose', 'frequency', 'region', 'division', 'owner_sex', 'is_local_client', 'has_card_before_loan']

X_all_raw = df_model[numeric_features + categorical_features].copy()
y_all = df_model['target'].copy()

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_all_raw, y_all, test_size=0.2, stratify=y_all, random_state=RANDOM_STATE
)
TRAIN_NULL_BEFORE = int(X_train_raw.isnull().sum().sum())
TEST_NULL_BEFORE = int(X_test_raw.isnull().sum().sum())
print(f'Train: {X_train_raw.shape} | Bad: {int(y_train.sum())}/{len(y_train)}')
print(f'Test:  {X_test_raw.shape} | Bad: {int(y_test.sum())}/{len(y_test)}')

## 2. Hạ tầng dùng chung

In [ ]:
def create_preprocessor(numeric_feats, categorical_feats, scaler=None, numeric_strategy='mean'):
    """scaler: None | 'standard' | 'robust'. Imputer luôn có để Pipeline tự xử lý null, không đụng DataFrame nguồn."""
    numeric_steps = [('imputer', SimpleImputer(strategy=numeric_strategy))]
    if scaler == 'standard':
        numeric_steps.append(('scaler', StandardScaler()))
    elif scaler == 'robust':
        numeric_steps.append(('scaler', RobustScaler()))
    numeric_transformer = Pipeline(steps=numeric_steps)

    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])
    return ColumnTransformer(transformers=[
        ('num', numeric_transformer, numeric_feats),
        ('cat', categorical_transformer, categorical_feats)
    ], remainder='drop')


def add_sampler_step(steps, imbalance_method):
    if imbalance_method == 'smote':
        steps.append(('sampler', SMOTE(random_state=RANDOM_STATE, k_neighbors=3)))
    elif imbalance_method == 'undersample':
        steps.append(('sampler', RandomUnderSampler(random_state=RANDOM_STATE)))
    return steps


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
SCORING = {
    'roc_auc': 'roc_auc', 'pr_auc': 'average_precision', 'accuracy': 'accuracy',
    'recall': 'recall', 'precision': 'precision', 'f1': 'f1',
}

results_all = []

def run_cv_pipeline(name, pipeline, X, y, cv_splitter=None):
    cv_res = cross_validate(pipeline, X, y, cv=(cv_splitter or skf), scoring=SCORING, n_jobs=-1, error_score='raise')
    row = {'technique': name}
    for k in SCORING:
        row[f'{k}_mean'] = cv_res[f'test_{k}'].mean()
        row[f'{k}_std'] = cv_res[f'test_{k}'].std()
    results_all.append(row)
    print(f"[{name}] PR-AUC={row['pr_auc_mean']:.4f}±{row['pr_auc_std']:.4f}  "
          f"ROC-AUC={row['roc_auc_mean']:.4f}  Acc={row['accuracy_mean']:.4f}  "
          f"Recall={row['recall_mean']:.4f}  Precision={row['precision_mean']:.4f}  F1={row['f1_mean']:.4f}")
    return row


def run_cv_pipeline_with_sample_weight(name, pipeline_template, X, y, sample_weight_full):
    """CV thủ công cho trường hợp cần sample_weight — không phụ thuộc tên tham số fit_params/params
    (đổi tên giữa các phiên bản scikit-learn), nên ổn định trên mọi phiên bản."""
    y_arr = y.values if hasattr(y, 'values') else np.asarray(y)
    fold_metrics = {k: [] for k in SCORING}
    for train_idx, val_idx in skf.split(X, y_arr):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y_arr[train_idx], y_arr[val_idx]
        sw_tr = sample_weight_full[train_idx]
        pipe = clone(pipeline_template)
        pipe.fit(X_tr, y_tr, clf__sample_weight=sw_tr)
        proba = pipe.predict_proba(X_val)[:, 1]
        pred = (proba >= 0.5).astype(int)
        fold_metrics['roc_auc'].append(roc_auc_score(y_val, proba))
        fold_metrics['pr_auc'].append(average_precision_score(y_val, proba))
        fold_metrics['accuracy'].append(accuracy_score(y_val, pred))
        fold_metrics['recall'].append(recall_score(y_val, pred, zero_division=0))
        fold_metrics['precision'].append(precision_score(y_val, pred, zero_division=0))
        fold_metrics['f1'].append(f1_score(y_val, pred, zero_division=0))
    row = {'technique': name}
    for k, vals in fold_metrics.items():
        row[f'{k}_mean'] = float(np.mean(vals))
        row[f'{k}_std'] = float(np.std(vals))
    results_all.append(row)
    print(f"[{name}] PR-AUC={row['pr_auc_mean']:.4f}±{row['pr_auc_std']:.4f}  "
          f"ROC-AUC={row['roc_auc_mean']:.4f}  Acc={row['accuracy_mean']:.4f}  "
          f"Recall={row['recall_mean']:.4f}  Precision={row['precision_mean']:.4f}  F1={row['f1_mean']:.4f}")
    return row


def manual_oof_proba(pipeline_template, X, y, sample_weight_full=None):
    """Xác suất out-of-fold để chọn threshold — không phụ thuộc fit_params/params của cross_val_predict."""
    y_arr = y.values if hasattr(y, 'values') else np.asarray(y)
    oof = np.zeros(len(y_arr), dtype=float)
    for train_idx, val_idx in skf.split(X, y_arr):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr = y_arr[train_idx]
        pipe = clone(pipeline_template)
        if sample_weight_full is not None:
            pipe.fit(X_tr, y_tr, clf__sample_weight=sample_weight_full[train_idx])
        else:
            pipe.fit(X_tr, y_tr)
        oof[val_idx] = pipe.predict_proba(X_val)[:, 1]
    return oof

print('Đã thiết lập skf, SCORING (đã có accuracy) và các hàm CV dùng chung.')

In [ ]:
def eval_and_plot_cm(name, pipeline, sample_weight=None, threshold=0.5, cmap='Blues'):
    """Fit lại pipeline trên toàn bộ train, đánh giá và vẽ confusion matrix
    kèm Accuracy / Precision / Recall / F1 / ROC-AUC / PR-AUC trên tiêu đề."""
    pipe = clone(pipeline)
    if sample_weight is not None:
        pipe.fit(X_train_raw, y_train, clf__sample_weight=sample_weight)
    else:
        pipe.fit(X_train_raw, y_train)

    proba = pipe.predict_proba(X_test_raw)[:, 1]
    pred = (proba >= threshold).astype(int)

    acc = accuracy_score(y_test, pred)
    prec = precision_score(y_test, pred, zero_division=0)
    rec = recall_score(y_test, pred, zero_division=0)
    f1 = f1_score(y_test, pred, zero_division=0)
    roc = roc_auc_score(y_test, proba)
    pr_auc = average_precision_score(y_test, proba)

    cm = confusion_matrix(y_test, pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay(cm, display_labels=['Good', 'Bad']).plot(
        ax=ax, cmap=cmap, values_format='d', colorbar=False
    )
    ax.set_title(
        f"{name}\n"
        f"Acc={acc:.4f} | Precision={prec:.4f} | Recall={rec:.4f} | "
        f"F1={f1:.4f} | ROC-AUC={roc:.4f} | PR-AUC={pr_auc:.4f}",
        fontsize=10, fontweight='bold'
    )
    plt.tight_layout()
    plt.show()

    return {
        'technique': name, 'accuracy': acc, 'precision': prec, 'recall': rec,
        'f1': f1, 'roc_auc': roc, 'pr_auc': pr_auc, 'threshold': threshold,
    }


print('Đã định nghĩa eval_and_plot_cm() - gọi hàm này sau mỗi lần train để vẽ confusion matrix kèm đủ chỉ số.')


## 3. Baseline (RF mặc định, class_weight='balanced')


Mốc tham chiếu: Random Forest 300 cây, xử lý mất cân bằng bằng `class_weight='balanced'` (scikit-learn tự tính trọng số nghịch đảo tần suất).

In [ ]:
baseline_pipe = ImbPipeline(steps=add_sampler_step(
    [('preprocessor', create_preprocessor(numeric_features, categorical_features, scaler=None))],
    imbalance_method='class_weight'
) + [('clf', RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1))])
baseline_row = run_cv_pipeline('0. Baseline (RF mặc định, class_weight)', baseline_pipe, X_train_raw, y_train)

In [ ]:
cm_baseline = eval_and_plot_cm('0. Baseline (RF mac dinh, class_weight)', baseline_pipe)

## 4. CẢI TIẾN 1 — Stacking RF + ExtraTrees → LogisticRegression


Một mô hình cây đơn lẻ vẫn có phương sai cao. Kết hợp 2 mô hình cây (RF và ExtraTrees) qua meta-learner Logistic Regression giúp giảm phương sai và bao quát vùng không gian đặc trưng tốt hơn.


- **Base Models**: Random Forest (300 cây, class_weight='balanced') + ExtraTrees (300 cây, class_weight='balanced')
- **Meta Model**: Logistic Regression (max_iter=1000)

In [ ]:
stacking_estimators = [
    ('rf', RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
    ('et', ExtraTreesClassifier(n_estimators=300, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
]
stacking_clf = StackingClassifier(
    estimators=stacking_estimators,
    final_estimator=LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    cv=5, passthrough=False, n_jobs=-1
)

pipe_imp2 = ImbPipeline(steps=[
    ('preprocessor', create_preprocessor(numeric_features, categorical_features, scaler=None)),
    ('clf', stacking_clf)
])
row_imp2 = run_cv_pipeline('1. Stacking RF+ExtraTrees → LR', pipe_imp2, X_train_raw, y_train)

In [ ]:
cm_imp2 = eval_and_plot_cm('1. Stacking RF+ExtraTrees -> LR', pipe_imp2)

## 5. CẢI TIẾN 2 — Weighted Random Forest (Bad:Good = 5:1)


`class_weight='balanced'` chưa đủ mạnh với mức mất cân bằng thật sự trong dữ liệu (Bad chiếm ~11%). Tăng trọng số lớp Bad lên 5 lần thủ công qua `sample_weight` — buộc mô hình "chú ý" hơn vào các khoản vay xấu.

In [ ]:
sample_weight_train_5to1 = np.where(y_train.values == 1, 5, 1.0)

pipe_imp1 = ImbPipeline(steps=[
    ('preprocessor', create_preprocessor(numeric_features, categorical_features, scaler=None)),
    ('clf', RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1))
])
row_imp1 = run_cv_pipeline_with_sample_weight(
    '2. Weighted RF (Bad:Good = 5:1)', pipe_imp1, X_train_raw, y_train, sample_weight_train_5to1
)

In [ ]:
cm_imp1 = eval_and_plot_cm('2. Weighted RF (Bad:Good = 5:1)', pipe_imp1, sample_weight=sample_weight_train_5to1)

## 6. CẢI TIẾN 3 — Feature Selection bằng SHAP (top-8)


Sử dụng SHAP (`TreeExplainer`) để đo mức đóng góp thực sự của từng đặc trưng. Loại bỏ các đặc trưng nhiễu, giữ lại đúng 8 đặc trưng quan trọng nhất — giúp mô hình bớt bị "phân tâm" bởi thông tin thừa.

In [ ]:
# Fit RF trên toàn bộ features để tính SHAP
preprocessor_for_shap = create_preprocessor(numeric_features, categorical_features, scaler=None)
X_train_transformed = preprocessor_for_shap.fit_transform(X_train_raw)
feature_names_transformed = preprocessor_for_shap.get_feature_names_out()

rf_for_shap = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
rf_for_shap.fit(X_train_transformed, y_train)

explainer = shap.TreeExplainer(rf_for_shap)
shap_values = explainer.shap_values(X_train_transformed)
if isinstance(shap_values, list):
    shap_vals_class1 = shap_values[1]
elif shap_values.ndim == 3:
    shap_vals_class1 = shap_values[:, :, 1]
else:
    shap_vals_class1 = shap_values

mean_abs_shap = np.abs(shap_vals_class1).mean(axis=0)
shap_importance = pd.Series(mean_abs_shap, index=feature_names_transformed).sort_values(ascending=False)
print('Top 15 đặc trưng quan trọng nhất theo SHAP:')
display(shap_importance.head(15))

In [ ]:
# =============================================
# BIỂU ĐỒ SHAP-1: SHAP Summary (beeswarm) — Random Forest (Bước 3, toàn bộ đặc trưng)
# =============================================
X_shap_dense_rf = X_train_transformed.toarray() if hasattr(X_train_transformed, 'toarray') else X_train_transformed
display_names_rf = [f.split('__', 1)[-1] for f in feature_names_transformed]

shap.summary_plot(
    shap_vals_class1, X_shap_dense_rf,
    feature_names=display_names_rf,
    max_display=15, show=False
)
fig = plt.gcf()
fig.set_size_inches(10, 6)
plt.title('SHAP Summary — Random Forest (Bước 3: chọn top-8 đặc trưng)', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'shap_summary_rf_step3.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Hàm ánh xạ tên đặc trưng đã transform về tên gốc
def map_transformed_to_original(transformed_name, num_feats, cat_feats):
    prefix, rest = transformed_name.split('__', 1)
    if prefix == 'num':
        return rest
    candidates = [c for c in cat_feats if rest == c or rest.startswith(c + '_')]
    return max(candidates, key=len) if candidates else rest

# Chọn top-8 features
top_8_transformed = shap_importance.head(8).index.tolist()
original_cols_8 = sorted(set(map_transformed_to_original(t, numeric_features, categorical_features) for t in top_8_transformed))
shap_num_8 = [c for c in original_cols_8 if c in numeric_features]
shap_cat_8 = [c for c in original_cols_8 if c in categorical_features]
print(f'SHAP top-8 → {len(shap_num_8)} numeric + {len(shap_cat_8)} categorical: {original_cols_8}')

pipe_imp3 = ImbPipeline(steps=[
    ('preprocessor', create_preprocessor(shap_num_8, shap_cat_8, scaler=None)),
    ('clf', RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1))
])
row_imp3 = run_cv_pipeline('3. SHAP top-8 (Feature Selection)', pipe_imp3, X_train_raw, y_train)

In [ ]:
cm_imp3 = eval_and_plot_cm('3. SHAP top-8 (Feature Selection)', pipe_imp3)

## 7. CẢI TIẾN 4 — LightGBM trên SHAP top-8


Gradient Boosting thường bắt được quan hệ phức tạp tốt hơn Random Forest. Đổi sang `LGBMClassifier` trên đúng tập 8 đặc trưng đã chọn ở bước trước.

In [ ]:
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

pipe_imp4 = Pipeline(steps=[
    ('preprocessor', create_preprocessor(shap_num_8, shap_cat_8, scaler=None)),
    ('clf', lgb.LGBMClassifier(
        scale_pos_weight=pos_weight,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1
    ))
])
row_imp4 = run_cv_pipeline('4. LightGBM SHAP top-8', pipe_imp4, X_train_raw, y_train)

In [ ]:
cm_imp4 = eval_and_plot_cm('4. LightGBM SHAP top-8', pipe_imp4)

## 8. CẢI TIẾN 5 — Optuna-tuned LightGBM (30 trials)


Tối ưu siêu tham số LightGBM bằng Bayesian Optimization (Optuna, TPE sampler). Tìm kiếm trên không gian: `learning_rate`, `max_depth`, `n_estimators`, `num_leaves`, `min_child_samples`, `min_split_gain`, `subsample`, `colsample_bytree`, `reg_alpha`, `reg_lambda`.

In [ ]:
def objective_lgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600, step=50),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150), # Tham số cốt lõi của LGBM
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100), # Tránh lá có quá ít dữ liệu
        'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 1.0), # Tương đương 'gamma' của XGBoost
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'scale_pos_weight': pos_weight,
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
        'verbose': -1 # Tắt các cảnh báo hệ thống của LightGBM
    }

    model = lgb.LGBMClassifier(**params)

    pipe = Pipeline(steps=[
        ('preprocessor', create_preprocessor(shap_num_8, shap_cat_8, scaler=None)),
        ('clf', model)
    ])

    # Ở đây bạn đang dùng scoring='f1' thay vì 'average_precision', rất hợp lý để tìm điểm cắt trực tiếp tốt nhất
    scores = cross_val_score(pipe, X_train_raw, y_train, cv=skf, scoring='f1', n_jobs=-1)
    return scores.mean()

study_lgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_lgb.optimize(objective_lgb, n_trials=30, show_progress_bar=False)

best_lgb_params = study_lgb.best_params

# Cập nhật lại các tham số cố định trước khi đưa vào mô hình final
best_lgb_params['scale_pos_weight'] = pos_weight
best_lgb_params['random_state'] = RANDOM_STATE
best_lgb_params['n_jobs'] = -1
best_lgb_params['verbose'] = -1

print(f'Siêu tham số tốt nhất (Optuna LightGBM): {best_lgb_params}')
print(f'F1 tốt nhất qua CV: {study_lgb.best_value:.4f}')

In [ ]:
print("--- ĐANG CHẠY: 5. Optuna-tuned LightGBM ---")

# Tạo pipeline từ best params và đánh giá đầy đủ
pipe_imp5 = Pipeline(steps=[
    ('preprocessor', create_preprocessor(shap_num_8, shap_cat_8, scaler=None)),
    ('clf', lgb.LGBMClassifier(**best_lgb_params))
])

row_imp5 = run_cv_pipeline('5. Optuna-tuned LightGBM', pipe_imp5, X_train_raw, y_train)

In [ ]:
cm_imp5 = eval_and_plot_cm('5. Optuna-tuned LightGBM', pipe_imp5)

## 9. CẢI TIẾN 6 — LightGBM SHAP top-12


LightGBM xử lý mất cân bằng tốt hơn XGBoost và có cơ chế phân nhánh leaf-wise (thay vì level-wise) tăng tính chính xác. Sử dụng đúng tập 12 đặc trưng đã chọn.

In [ ]:
import shap
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline

print("="*70)
print("BƯỚC 6: TÌM KIẾM VÀ ÁP DỤNG TOP-12 ĐẶC TRƯNG SHAP CHO LIGHTGBM")
print("="*70)

# 1. Tính toán SHAP values trên mô hình LightGBM
preprocessor_for_shap = create_preprocessor(numeric_features, categorical_features, scaler=None)
X_train_transformed = preprocessor_for_shap.fit_transform(X_train_raw)
feature_names_transformed = preprocessor_for_shap.get_feature_names_out()

# Huấn luyện mô hình thuần trên dữ liệu đã transform để lấy gốc tính SHAP
# Cấu hình này khớp với tham số bạn đang thiết lập cho pipe_imp7
lgb_for_shap = lgb.LGBMClassifier(
    scale_pos_weight=pos_weight,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)
lgb_for_shap.fit(X_train_transformed, y_train)

# Tính SHAP
explainer = shap.TreeExplainer(lgb_for_shap)
shap_values = explainer.shap_values(X_train_transformed)

# Xử lý format trả về của SHAP
if isinstance(shap_values, list):
    shap_vals_target = shap_values[1]
elif len(shap_values.shape) == 3:
    shap_vals_target = shap_values[:, :, 1]
else:
    shap_vals_target = shap_values

mean_abs_shap = np.abs(shap_vals_target).mean(axis=0)
shap_importance = pd.Series(mean_abs_shap, index=feature_names_transformed).sort_values(ascending=False)

# Hàm map ngược tên đặc trưng (đã biến đổi One-Hot) về tên cột nguyên thủy
def map_transformed_to_original(transformed_name, num_feats, cat_feats):
    prefix, rest = transformed_name.split('__', 1)
    if prefix == 'num':
        return rest
    candidates = [c for c in cat_feats if rest == c or rest.startswith(c + '_')]
    return max(candidates, key=len) if candidates else rest

# 2. Trích xuất chính xác Top-12 đặc trưng
top_12_transformed = shap_importance.head(12).index.tolist()
original_cols_12 = sorted(set(map_transformed_to_original(t, numeric_features, categorical_features) for t in top_12_transformed))

shap_num_12 = [c for c in original_cols_12 if c in numeric_features]
shap_cat_12 = [c for c in original_cols_12 if c in categorical_features]

print(f"Top 12 SHAP Numeric: {shap_num_12}")
print(f"Top 12 SHAP Categorical: {shap_cat_12}\n")

# 3. Tạo Pipeline và Đánh giá (Bước 7 theo mạch sơ đồ)
pipe_imp7 = Pipeline(steps=[
    ('preprocessor', create_preprocessor(shap_num_12, shap_cat_12, scaler=None)),
    ('clf', lgb.LGBMClassifier(
        scale_pos_weight=pos_weight,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1
    ))
])

print("--- ĐANG CHẠY: 6. LightGBM SHAP top-12 ---")
row_imp7 = run_cv_pipeline('6. LightGBM SHAP top-12', pipe_imp7, X_train_raw, y_train)


In [ ]:
cm_imp7 = eval_and_plot_cm('6. LightGBM SHAP top-12', pipe_imp7)

In [ ]:
# =============================================
# BIỂU ĐỒ SHAP-2: SHAP Summary (beeswarm) — LightGBM (Bước 6, toàn bộ đặc trưng)
# =============================================
X_shap_dense_lgb = X_train_transformed.toarray() if hasattr(X_train_transformed, 'toarray') else X_train_transformed
display_names_lgb = [f.split('__', 1)[-1] for f in feature_names_transformed]

shap.summary_plot(
    shap_vals_target, X_shap_dense_lgb,
    feature_names=display_names_lgb,
    max_display=15, show=False
)
fig = plt.gcf()
fig.set_size_inches(10, 6)
plt.title('SHAP Summary — LightGBM (Bước 6: chọn top-12 đặc trưng)', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'shap_summary_lgb_step6.png'), dpi=150, bbox_inches='tight')
plt.show()


## 10. CẢI TIẾN 7 — Optuna tune XGBoost SHAP top-12


Mô hình XGBoost sau khi được tune có khả năng tiêu hóa thêm thông tin mà không bị overfit quá đà. Mở rộng số đặc trưng từ top 8 lên **top 12** (lấy từ bảng SHAP ở bước 3).

In [ ]:
import shap
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline

print("="*70)
print("BƯỚC 7: TÌM KIẾM VÀ ÁP DỤNG TOP-12 ĐẶC TRƯNG SHAP CHO XGBOOST")
print("="*70)

# ── Bước 7a: Optuna tune XGBoost (30 trials, tối ưu F1) ──
def objective_xgb(trial):
    params = {
        'n_estimators':    trial.suggest_int('n_estimators', 100, 600, step=50),
        'learning_rate':   trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth':       trial.suggest_int('max_depth', 3, 10),
        'colsample_bytree':trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'subsample':       trial.suggest_float('subsample', 0.6, 1.0),
        'min_child_weight':trial.suggest_int('min_child_weight', 1, 10),
        'gamma':           trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha':       trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':      trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }
    model = xgb.XGBClassifier(
        **params,
        scale_pos_weight=pos_weight,
        eval_metric='logloss',
        use_label_encoder=False,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    pipe = Pipeline(steps=[
        ('preprocessor', create_preprocessor(shap_num_8, shap_cat_8, scaler=None)),
        ('clf', model)
    ])
    scores = cross_val_score(pipe, X_train_raw, y_train, cv=skf, scoring='f1', n_jobs=-1)
    return scores.mean()

study_xgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_xgb.optimize(objective_xgb, n_trials=30, show_progress_bar=False)

best_xgb_params = study_xgb.best_params
print(f'Siêu tham số tốt nhất (Optuna XGBoost): {best_xgb_params}')
print(f'F1 tốt nhất qua CV (XGBoost): {study_xgb.best_value:.4f}')

# 1. Tính toán SHAP values trên mô hình XGBoost tốt nhất (từ cấu hình best_xgb_params)
preprocessor_for_shap_xgb = create_preprocessor(numeric_features, categorical_features, scaler=None)
X_train_transformed_xgb = preprocessor_for_shap_xgb.fit_transform(X_train_raw)
feature_names_transformed_xgb = preprocessor_for_shap_xgb.get_feature_names_out()

# Huấn luyện mô hình thuần trên dữ liệu đã transform để lấy gốc tính SHAP
xgb_for_shap = xgb.XGBClassifier(
    **best_xgb_params,
    scale_pos_weight=pos_weight,
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
xgb_for_shap.fit(X_train_transformed_xgb, y_train)

# Tính SHAP
explainer_xgb = shap.TreeExplainer(xgb_for_shap)
shap_values_xgb = explainer_xgb.shap_values(X_train_transformed_xgb)

# Xử lý format trả về của SHAP (XGBoost thường trả về mảng log-odds)
if isinstance(shap_values_xgb, list):
    shap_vals_target_xgb = shap_values_xgb[1]
elif len(shap_values_xgb.shape) == 3:
    shap_vals_target_xgb = shap_values_xgb[:, :, 1]
else:
    shap_vals_target_xgb = shap_values_xgb

mean_abs_shap_xgb = np.abs(shap_vals_target_xgb).mean(axis=0)
shap_importance_xgb = pd.Series(mean_abs_shap_xgb, index=feature_names_transformed_xgb).sort_values(ascending=False)

# Hàm map ngược tên đặc trưng (đã biến đổi One-Hot) về tên cột nguyên thủy
def map_transformed_to_original(transformed_name, num_feats, cat_feats):
    prefix, rest = transformed_name.split('__', 1)
    if prefix == 'num':
        return rest
    candidates = [c for c in cat_feats if rest == c or rest.startswith(c + '_')]
    return max(candidates, key=len) if candidates else rest

# 2. Mở rộng feature set từ 8 → 12 (Trích xuất Top-12 đặc trưng)
top_12_transformed = shap_importance_xgb.head(12).index.tolist()
original_cols_12 = sorted(set(map_transformed_to_original(t, numeric_features, categorical_features) for t in top_12_transformed))

shap_num_12 = [c for c in original_cols_12 if c in numeric_features]
shap_cat_12 = [c for c in original_cols_12 if c in categorical_features]

print(f'SHAP top-12 → {len(shap_num_12)} numeric + {len(shap_cat_12)} categorical: {original_cols_12}\n')

# 3. Tạo Pipeline và Đánh giá (Đúng nguyên mẫu code của bạn)
pipe_imp6 = Pipeline(steps=[
    ('preprocessor', create_preprocessor(shap_num_12, shap_cat_12, scaler=None)),
    ('clf', xgb.XGBClassifier(
        **best_xgb_params,
        scale_pos_weight=pos_weight,
        eval_metric='logloss',
        use_label_encoder=False,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

print("--- ĐANG CHẠY: 7. XGBoost SHAP top-12 ---")
row_imp6 = run_cv_pipeline('7. XGBoost SHAP top-12', pipe_imp6, X_train_raw, y_train)
# Lưu lại pipeline tốt nhất trước threshold
best_pre_threshold_pipeline = clone(pipe_imp7)
best_pre_threshold_name = '7. XGBoost SHAP top-12'
best_shap_num = shap_num_12
best_shap_cat = shap_cat_12
# (Tùy chọn) Ghi đè vào kết quả để bảng tổng hợp không bị loãng
# results_all = [row for row in results_all if not str(row['technique']).startswith('7. LightGBM')]
# results_all.append(row_imp7)

In [ ]:
cm_imp6 = eval_and_plot_cm('7. XGBoost SHAP top-12', pipe_imp6)

In [ ]:
# =============================================
# BIỂU ĐỒ SHAP-3: SHAP Summary (beeswarm) — XGBoost (Bước 7, toàn bộ đặc trưng)
# =============================================
X_shap_dense_xgb = X_train_transformed_xgb.toarray() if hasattr(X_train_transformed_xgb, 'toarray') else X_train_transformed_xgb
display_names_xgb = [f.split('__', 1)[-1] for f in feature_names_transformed_xgb]

shap.summary_plot(
    shap_vals_target_xgb, X_shap_dense_xgb,
    feature_names=display_names_xgb,
    max_display=15, show=False
)
fig = plt.gcf()
fig.set_size_inches(10, 6)
plt.title('SHAP Summary — XGBoost (Bước 7: chọn top-12 đặc trưng)', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'shap_summary_xgb_step7.png'), dpi=150, bbox_inches='tight')
plt.show()


## 11. CẢI TIẾN 8 — RealMLP (SHAP top-12) + Threshold Optimization (F1-optimal)

*(Đã thay mô hình ở bước này: thay vì tối ưu ngưỡng trên **Optuna-tuned XGBoost SHAP top-12**, ta huấn luyện **RealMLP** — kiến trúc MLP "tuned-default" cho dữ liệu dạng bảng từ thư viện `pytabkit` — trên đúng tập 12 đặc trưng SHAP đã chọn ở Bước 7, rồi tối ưu ngưỡng trên xác suất OOF của RealMLP. RealMLP tự thực hiện chuẩn hoá/embedding nội bộ nên pipeline tiền xử lý ở bước này chỉ cần impute giá trị thiếu.)*

Bước cuối: thay vì dùng ngưỡng phân loại mặc định 0.5, ta tìm ngưỡng tối ưu hóa F1 Score trên xác suất out-of-fold (OOF) của tập TRAIN — tránh rò rỉ vào Test.

Tính 3 loại ngưỡng:
- **F1-optimal**: Tối đa hóa F1
- **F-beta (β=1.15)**: Coi trọng recall hơn precision một chút (hợp lý cho bài toán tín dụng)
- **Cost-optimal** (FN=5x, FP=1x): Bỏ lọt nợ xấu tốn kém hơn từ chối nhầm khách tốt

In [ ]:
from sklearn.metrics import precision_recall_curve, confusion_matrix, accuracy_score, recall_score, precision_score, f1_score
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import cross_validate
import numpy as np
import pandas as pd

print("=" * 70)
print("BƯỚC 8: RealMLP (SHAP top-12) + TỐI ƯU NGƯỠNG (THRESHOLD)")
print("=" * 70)

# 0. Đảm bảo có pytabkit (thư viện chứa RealMLP) -----------------------------
try:
    from pytabkit import RealMLP_TD_Classifier
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytabkit'], check=True)
    from pytabkit import RealMLP_TD_Classifier


# 1. Wrapper sklearn cho RealMLP ---------------------------------------------
# RealMLP-TD tự làm tiền xử lý nội bộ (one-hot cho hạng mục nhỏ, embedding cho
# hạng mục lớn, robust-scale cho số liệu...) nên wrapper này chỉ cần đưa vào
# một DataFrame đã impute (không one-hot, không scale thủ công) + danh sách
# tên cột categorical (cat_col_names) để RealMLP tự xử lý đúng cách.
# LƯU Ý thứ tự kế thừa: ClassifierMixin PHẢI đứng trước BaseEstimator để
# sklearn (is_classifier / __sklearn_tags__) nhận diện đúng đây là classifier.
class RealMLPClassifierWrapper(ClassifierMixin, BaseEstimator):
    def __init__(self, cat_col_names=None, random_state=RANDOM_STATE, n_epochs=256, device=None):
        self.cat_col_names = cat_col_names
        self.random_state = random_state
        self.n_epochs = n_epochs
        self.device = device

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        X_df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        self.model_ = RealMLP_TD_Classifier(
            random_state=self.random_state,
            n_epochs=self.n_epochs,
            device=self.device,
            verbosity=0,
        )
        self.model_.fit(X_df, np.asarray(y), cat_col_names=self.cat_col_names)
        return self

    def predict_proba(self, X):
        X_df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        return self.model_.predict_proba(X_df)

    def predict(self, X):
        proba = self.predict_proba(X)
        return self.classes_[np.argmax(proba, axis=1)]


def create_preprocessor_realmlp(numeric_feats, categorical_feats):
    """Preprocessor RIÊNG cho RealMLP: chỉ impute giá trị thiếu (numeric: mean,
    categorical: most_frequent), KHÔNG one-hot / KHÔNG scale (RealMLP tự làm
    việc đó nội bộ). set_output('pandas') + verbose_feature_names_out=False để
    giữ nguyên tên cột gốc, dùng làm cat_col_names cho RealMLP."""
    numeric_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='mean'))])
    categorical_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent'))])
    ct = ColumnTransformer(transformers=[
        ('num', numeric_transformer, numeric_feats),
        ('cat', categorical_transformer, categorical_feats),
    ], remainder='drop', verbose_feature_names_out=False)
    ct.set_output(transform='pandas')
    return ct


# 2. Pipeline RealMLP trên đúng tập SHAP top-12 (shap_num_12 / shap_cat_12, lấy từ Bước 7)
pipe_imp8_realmlp = Pipeline(steps=[
    ('preprocessor', create_preprocessor_realmlp(shap_num_12, shap_cat_12)),
    ('clf', RealMLPClassifierWrapper(cat_col_names=shap_cat_12, random_state=RANDOM_STATE, n_epochs=256)),
])

# 3. Đánh giá CV của riêng RealMLP (PR-AUC / ROC-AUC không đổi theo threshold)
#    -> tính trực tiếp bằng cross_validate, KHÔNG dùng run_cv_pipeline để
#    tránh tạo thêm một hàng "model" riêng trong results_all (giữ đúng cấu
#    trúc 9 mốc: Baseline + 8 cải tiến, mốc 8 = RealMLP + Threshold gộp chung).
print("--- ĐANG CHẠY: 8. RealMLP SHAP top-12 (Cross-Validation) ---")
cv_res_realmlp = cross_validate(
    pipe_imp8_realmlp, X_train_raw, y_train, cv=skf, scoring=SCORING, n_jobs=1, error_score='raise'
)
base_pr_auc = cv_res_realmlp['test_pr_auc'].mean()
base_pr_std = cv_res_realmlp['test_pr_auc'].std()
base_roc_auc = cv_res_realmlp['test_roc_auc'].mean()
print(f"[8. RealMLP SHAP top-12] PR-AUC={base_pr_auc:.4f}±{base_pr_std:.4f}  "
      f"ROC-AUC={base_roc_auc:.4f}  F1@0.5={cv_res_realmlp['test_f1'].mean():.4f}")

# 4. Xác suất OOF bằng pipeline RealMLP (để chọn ngưỡng, tránh rò rỉ vào Test)
oof_proba_realmlp = manual_oof_proba(pipe_imp8_realmlp, X_train_raw, y_train)

# 5. Tính toán đường cong Precision-Recall
precisions, recalls, pr_thresholds = precision_recall_curve(y_train, oof_proba_realmlp)

# --- F1-optimal ---
f1_per_thr = 2 * precisions * recalls / (precisions + recalls + 1e-12)
threshold_f1_optimal = pr_thresholds[int(np.argmax(f1_per_thr[:-1]))]

# --- F-beta optimal (β=1.15) ---
BETA = 1.15
fbeta_per_thr = (1 + BETA**2) * precisions * recalls / (BETA**2 * precisions + recalls + 1e-12)
threshold_fbeta_optimal = pr_thresholds[int(np.argmax(fbeta_per_thr[:-1]))]

# --- Cost-optimal (FN tốn 5x, FP tốn 1x) ---
COST_FN, COST_FP = 5, 1
cost_per_thr = []
for thr in pr_thresholds:
    pred = (oof_proba_realmlp >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_train, pred).ravel()
    cost_per_thr.append(fn * COST_FN + fp * COST_FP)
threshold_cost_optimal = pr_thresholds[int(np.argmin(cost_per_thr))]

print(f'Ngưỡng F1-optimal               : {threshold_f1_optimal:.4f}')
print(f'Ngưỡng F-beta (β={BETA}) optimal : {threshold_fbeta_optimal:.4f}')
print(f'Ngưỡng Cost-optimal (FN={COST_FN}x, FP={COST_FP}x): {threshold_cost_optimal:.4f}\n')

# 6. Chọn ngưỡng F1-optimal làm ngưỡng triển khai chính (theo yêu cầu)
FINAL_THRESHOLD = threshold_f1_optimal

# Đánh giá các ngưỡng trên OOF
thresholds_to_eval = [
    ('8a. Threshold RealMLP: F1-optimal', threshold_f1_optimal),
    (f'8b. Threshold RealMLP: F-beta(β={BETA})-optimal', threshold_fbeta_optimal),
    ('8c. Threshold RealMLP: Cost-optimal', threshold_cost_optimal),
]

# Loại bỏ kết quả threshold cũ (XGBoost hoặc RealMLP của lần chạy trước) nếu
# chạy lại cell để bảng không bị trùng lặp
results_all = [row for row in results_all if not str(row['technique']).startswith('8.')]

print("--- KIỂM TRA HIỆU SUẤT TRÊN CÁC NGƯỠNG ---")
for label, thr in thresholds_to_eval:
    pred = (oof_proba_realmlp >= thr).astype(int)
    f1_val = f1_score(y_train, pred)
    print(f'[{label}] thr={thr:.4f}  Acc={accuracy_score(y_train, pred):.4f}  '
          f'Recall={recall_score(y_train, pred):.4f}  Precision={precision_score(y_train, pred):.4f}  '
          f'F1={f1_val:.4f}')

# 7. Thêm kết quả RealMLP + threshold F1-optimal (ngưỡng được chọn) vào bảng tổng hợp chung
pred_final = (oof_proba_realmlp >= FINAL_THRESHOLD).astype(int)
results_all.append({
    'technique': f'8. RealMLP SHAP top-12 + Threshold F1-optimal (thr={FINAL_THRESHOLD:.4f})',
    'pr_auc_mean': base_pr_auc,
    'pr_auc_std': base_pr_std,
    'roc_auc_mean': base_roc_auc,
    'accuracy_mean': accuracy_score(y_train, pred_final),
    'recall_mean': recall_score(y_train, pred_final),
    'precision_mean': precision_score(y_train, pred_final),
    'f1_mean': f1_score(y_train, pred_final),
})

print(f'\n=> Ngưỡng triển khai RealMLP (F1-optimal): {FINAL_THRESHOLD:.4f}')
print('LƯU Ý: pytabkit.RealMLP_TD_Classifier không hỗ trợ sample_weight trực tiếp '
      'trong fit(), nên việc xử lý mất cân bằng lớp ở bước này dựa hoàn toàn vào '
      'việc tối ưu ngưỡng quyết định (threshold) thay vì trọng số mẫu.')

In [ ]:
cm_imp8_thr = eval_and_plot_cm('8. RealMLP SHAP top-12', pipe_imp8_realmlp)

In [ ]:
import numpy as np

# Calculate combined score and its std for each entry in results_all
for row in results_all:
    f1_mean = row['f1_mean']
    pr_auc_mean = row['pr_auc_mean']
    f1_std = row['f1_std'] if 'f1_std' in row else 0.0  # Handle cases where std might be missing
    pr_auc_std = row['pr_auc_std'] if 'pr_auc_std' in row else 0.0 # Handle cases where std might be missing

    # Calculate combined mean score
    combined_mean = (0.6 * float(f1_mean)) + (0.4 * float(pr_auc_mean))
    row['combined_0.6F1+0.4PRAUC_mean'] = combined_mean

    # Calculate combined standard deviation using error propagation
    # Assuming F1 and PR-AUC are independent for simplicity
    combined_std = np.sqrt((0.6 * float(f1_std))**2 + (0.4 * float(pr_auc_std))**2)
    row['combined_0.6F1+0.4PRAUC_std'] = combined_std

print('Đã tính toán thêm metric tổng hợp (0.6*F1 + 0.4*PR-AUC) cho tất cả các bước.')

In [ ]:
# ============================================================================
# [ĐÃ VÔ HIỆU HÓA] Cell này là bản nháp/trùng lặp cũ của BƯỚC 7 & 8
# (tính lại SHAP top-12 + Optuna tune XGBoost 30 trials + threshold optimization
# cho XGBoost), bị bỏ sót lại trong notebook và sẽ GHI ĐÈ kết quả của Bước 8
# (giờ đã là RealMLP, xem cell phía trên) nếu được chạy.
#
# Đã comment out toàn bộ để tránh:
#   1) chạy lại Optuna 30 trials một cách dư thừa (tốn thời gian),
#   2) vô tình thay kết quả "8. RealMLP SHAP top-12 + Threshold..." trong
#      results_all trở lại bằng XGBoost.
#
# Nếu vì lý do nào đó bạn muốn xem lại phiên bản XGBoost + Threshold (bản cũ
# trước khi thay bằng RealMLP), có thể bỏ comment toàn bộ khối bên dưới.
# ============================================================================

# import shap
# import optuna
# import numpy as np
# import pandas as pd
# import xgboost as xgb
# from sklearn.pipeline import Pipeline
# from sklearn.base import clone
# from sklearn.metrics import precision_recall_curve, confusion_matrix, accuracy_score, recall_score, precision_score, f1_score
# from sklearn.model_selection import cross_val_score
#
# print("=" * 80)
# print("BƯỚC 7 & 8: TỐI ƯU TOÀN DIỆN XGBOOST SHAP TOP-12 + THRESHOLD (LEGACY - ĐÃ THAY BẰNG REALMLP)")
# print("=" * 80)
# ... (nội dung gốc đã được vô hiệu hóa, xem lịch sử notebook nếu cần khôi phục)

print("Cell legacy (XGBoost SHAP top-12 + Threshold, bản trùng lặp) đã bị vô hiệu hóa "
      "— Bước 8 hiện dùng RealMLP, xem các cell phía trên.")

## 12. Tổng hợp bảng kết quả & biểu đồ


Bảng kết quả toàn bộ 9 mốc (Baseline + 8 cải tiến) và biểu đồ trực quan hóa đường cong F1 tăng dần.

In [ ]:
# Bảng tổng hợp
print("\n=== BẢNG XẾP HẠNG TỔNG HỢP MỚI CẬP NHẬT ===")
comparison_df_new = pd.DataFrame(results_all).sort_values(
    by=['f1_mean'], ascending=[False]
).reset_index(drop=True)

display(comparison_df_new[['technique', 'pr_auc_mean', 'pr_auc_std', 'roc_auc_mean', 'accuracy_mean',
                           'recall_mean', 'precision_mean', 'f1_mean']])

In [ ]:
# Bảng tổng hợp
print("\n=== BẢNG XẾP HẠNG TỔNG HỢP MỚI CẬP NHẬT ===")
comparison_df_new = pd.DataFrame(results_all).sort_values(
    by=['combined_0.6F1+0.4PRAUC_mean'], ascending=[False]
).reset_index(drop=True)

display(comparison_df_new[['technique', 'pr_auc_mean', 'pr_auc_std', 'roc_auc_mean', 'accuracy_mean',
                           'recall_mean', 'precision_mean', 'f1_mean', 'combined_0.6F1+0.4PRAUC_mean', 'combined_0.6F1+0.4PRAUC_std']])

In [ ]:
# =============================================
# BIỂU ĐỒ 1: Đường cong F1 tăng dần qua 9 mốc
# =============================================
step_labels = [
    'Baseline',
    '1. Stacking',
    '2. Weighted RF',
    '3. SHAP top-8',
    '4. LightGBM top-8',
    '5. Optuna LGB',
    '6. LGB top-12',
    '7. XGB top-12',
    '8. RealMLP+Thresh'
]
f1_values = [row['f1_mean'] for row in results_all]

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(range(len(f1_values)), f1_values, color=plt.cm.viridis(np.linspace(0.2, 0.9, len(f1_values))),
              edgecolor='white', linewidth=1.5)
ax.plot(range(len(f1_values)), f1_values, 'o-', color='#FF6B6B', linewidth=2.5, markersize=8, zorder=5)

# Gắn nhãn giá trị
for i, (bar, val) in enumerate(zip(bars, f1_values)):
    ax.text(i, val + 0.008, f'{val:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=9)
    if i > 0:
        delta = val - f1_values[i-1]
        ax.text(i, val - 0.025, f'+{delta:.4f}', ha='center', va='top', fontsize=7, color='green', fontstyle='italic')

ax.set_xticks(range(len(step_labels)))
ax.set_xticklabels(step_labels, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('F1 Score (Cross-Validation)', fontsize=12)
ax.set_title('Đường cong F1 qua 8 bước cải tiến — Diminishing Returns', fontsize=14, fontweight='bold')
ax.set_ylim(0.4, max(f1_values) + 0.06)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'f1_improvement_chart.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Đã lưu biểu đồ F1 tại:', os.path.join(MODEL_DIR, 'f1_improvement_chart.png'))

### Đường cong PR-AUC qua 9 mốc

In [ ]:
pr_auc_values = [row['pr_auc_mean'] for row in results_all]
pr_auc_stds = [row['pr_auc_std'] for row in results_all]

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(range(len(pr_auc_values)), pr_auc_values, yerr=pr_auc_stds, capsize=5, color=plt.cm.cividis(np.linspace(0.2, 0.9, len(pr_auc_values))), edgecolor='white', linewidth=1.5)
ax.plot(range(len(pr_auc_values)), pr_auc_values, 'o-', color='#FF6B6B', linewidth=2.5, markersize=8, zorder=5)

# Gắn nhãn giá trị
for i, (bar, val) in enumerate(zip(bars, pr_auc_values)):
    ax.text(i, val + 0.008, f'{val:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=9)
    if i > 0:
        delta = val - pr_auc_values[i-1]
        ax.text(i, val - 0.025, f'{delta:+.4f}', ha='center', va='top', fontsize=7, color='green' if delta > 0 else 'red', fontstyle='italic')

ax.set_xticks(range(len(step_labels)))
ax.set_xticklabels(step_labels, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('PR-AUC Score (Cross-Validation)', fontsize=12)
ax.set_title('Đường cong PR-AUC qua 8 bước cải tiến — Diminishing Returns', fontsize=14, fontweight='bold')
ax.set_ylim(0.4, max(pr_auc_values) + 0.06)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'pr_auc_improvement_chart.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Đã lưu biểu đồ PR-AUC tại:', os.path.join(MODEL_DIR, 'pr_auc_improvement_chart.png'))

### Đường cong (0.6*F1 + 0.4*PR-AUC) tăng dần qua 9 bước cải tiến

In [ ]:
combined_values = [row['combined_0.6F1+0.4PRAUC_mean'] for row in results_all]
combined_stds = [row['combined_0.6F1+0.4PRAUC_std'] for row in results_all]

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(range(len(combined_values)), combined_values, yerr=combined_stds, capsize=5,
              color=plt.cm.coolwarm(np.linspace(0.2, 0.9, len(combined_values))),
              edgecolor='white', linewidth=1.5)
ax.plot(range(len(combined_values)), combined_values, 'o-', color='#3CB371', linewidth=2.5, markersize=8, zorder=5)

# Gắn nhãn giá trị
for i, (bar, val) in enumerate(zip(bars, combined_values)):
    ax.text(i, val + 0.008, f'{val:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=9)
    if i > 0:
        delta = val - combined_values[i-1]
        ax.text(i, val - 0.025, f'{delta:+.4f}', ha='center', va='top', fontsize=7, color='green' if delta > 0 else 'red', fontstyle='italic')

ax.set_xticks(range(len(step_labels)))
ax.set_xticklabels(step_labels, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Combined Score (0.6*F1 + 0.4*PR-AUC) (Cross-Validation)', fontsize=12)
ax.set_title('Đường cong Combined Score qua 8 bước cải tiến — Diminishing Returns', fontsize=14, fontweight='bold')
ax.set_ylim(0.4, max(combined_values) + 0.06)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'combined_score_improvement_chart.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Đã lưu biểu đồ Combined Score tại:', os.path.join(MODEL_DIR, 'combined_score_improvement_chart.png'))

In [ ]:
# ============================================================
# BƯỚC 9: XÁC ĐỊNH MODEL TỐT NHẤT TRÊN CV TRƯỚC KHI TIẾP XÚC TEST
# ============================================================
import numpy as np

# Bảng ánh xạ: tên kỹ thuật → pipeline đã định nghĩa
# (Chỉ bao gồm các model CÓ pipeline; các row "Threshold-only" của những kỹ
#  thuật chỉ hậu xử lý trên một model đã liệt kê thì KHÔNG được thêm vào đây.)
candidate_models = {
    '0. Baseline (RF mặc định, class_weight)':        baseline_pipe,
    '1. Stacking RF+ExtraTrees → LR':                 pipe_imp2,
    '2. Weighted RF (Bad:Good = 5:1)':                pipe_imp1,
    '3. SHAP top-8 (Feature Selection)':              pipe_imp3,
    '4. LightGBM SHAP top-8':                         pipe_imp4,
    '5. Optuna-tuned LightGBM':                       pipe_imp5,
    '6. LightGBM SHAP top-12':                        pipe_imp7,
    '7. XGBoost SHAP top-12':                         pipe_imp6,
}

# Bước 8 (RealMLP + Threshold) là MỘT MODEL THẬT SỰ KHÁC (không phải hậu xử lý
# trên model nào đã liệt kê ở trên), nên cũng được đưa vào danh sách ứng viên
# để cạnh tranh công bằng. Dùng đúng tên kỹ thuật động (có kèm ngưỡng) đã được
# ghi trong results_all ở cell Bước 8, và map sang pipeline RealMLP CHƯA fit.
candidate_models[results_all[-1]['technique']] = pipe_imp8_realmlp

# Lọc results_all chỉ lấy các model có pipeline (loại bỏ các row threshold
# thuần khác nếu có)
cv_rows = [r for r in results_all if r['technique'] in candidate_models]

# Tiêu chí 1: Combined Score = 0.6 * F1 + 0.4 * PR-AUC (đã tính ở Bước 11)
best_row_combined = max(cv_rows, key=lambda r: r['combined_0.6F1+0.4PRAUC_mean'])

# Tiêu chí 2: F1 thuần (tiebreaker nếu cần)
best_row_f1 = max(cv_rows, key=lambda r: r['f1_mean'])

print("=" * 70)
print("KẾT QUẢ CHỌN MODEL TỐT NHẤT (CHỈ DỰA TRÊN CV — CHƯA DÙNG TEST)")
print("=" * 70)

print("\n[Tiêu chí chính: Combined Score = 0.6×F1 + 0.4×PR-AUC]")
for r in sorted(cv_rows, key=lambda r: r['combined_0.6F1+0.4PRAUC_mean'], reverse=True):
    marker = " ← WINNER" if r['technique'] == best_row_combined['technique'] else ""
    print(f"  {r['technique']:<45s}  "
          f"Combined={r['combined_0.6F1+0.4PRAUC_mean']:.4f}  "
          f"F1={r['f1_mean']:.4f}  PR-AUC={r['pr_auc_mean']:.4f}{marker}")

# Xác nhận pipeline tương ứng
best_pre_threshold_name = best_row_combined['technique']
best_pipeline = candidate_models[best_pre_threshold_name]

# Cả XGBoost (Bước 7) lẫn RealMLP (Bước 8) đều dùng chung tập đặc trưng
# SHAP top-12 (shap_num_12 / shap_cat_12). FINAL_THRESHOLD hiện tại là ngưỡng
# F1-optimal được tính ở Bước 8 (trên OOF của RealMLP). Nếu model thắng cuộc
# không phải RealMLP, ngưỡng 0.5 mặc định cũng vẫn được báo cáo song song ở
# Section 13 (xem y_test_pred_default) để đối chiếu.
best_shap_num = shap_num_12
best_shap_cat = shap_cat_12

print(f"\n✅ Model được chọn : {best_pre_threshold_name}")
print(f"   F1 (CV)         : {best_row_combined['f1_mean']:.4f} ± {best_row_combined.get('f1_std', 0):.4f}")
print(f"   PR-AUC (CV)     : {best_row_combined['pr_auc_mean']:.4f} ± {best_row_combined['pr_auc_std']:.4f}")
print(f"   Combined Score  : {best_row_combined['combined_0.6F1+0.4PRAUC_mean']:.4f}")
print(f"   ROC-AUC (CV)    : {best_row_combined['roc_auc_mean']:.4f}")
print(f"\n⛔ Tập Test CHƯA được sử dụng ở bước này.")
print(f"   → Tiếp tục fit lại trên toàn bộ Train và đánh giá Test ở Section 13.")

In [ ]:
for r in results_all:
    print(repr(r['technique']))

## 13. Đánh giá cuối trên tập Test giữ riêng


Fit mô hình tốt nhất (đã xác định ở Bước 9 — `best_pipeline`, có thể là RealMLP, XGBoost, LightGBM... tùy kết quả CV) trên **toàn bộ tập Train**, dự đoán trên tập Test, áp dụng ngưỡng F1-optimal. Đây là lần **duy nhất** tập Test được sử dụng.

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, confusion_matrix
from sklearn.base import clone
import pandas as pd

print("="*70)
print(f"ĐÁNH GIÁ MÔ HÌNH CHIẾN THẮNG TRÊN TẬP TEST ({best_pre_threshold_name})")
print("="*70)

# 1. Khởi tạo và huấn luyện lại trên toàn bộ tập Train
# Dùng best_pipeline đã xác định ở Bước 9 (không hard-code nữa)
final_pipeline = clone(best_pipeline)
final_pipeline.fit(X_train_raw, y_train)

# 2. Tính xác suất dự đoán trên tập Test
y_test_proba = final_pipeline.predict_proba(X_test_raw)[:, 1]

# 3. Phân lớp dựa trên 2 ngưỡng: Mặc định (0.5) và Tối ưu F1 (FINAL_THRESHOLD)
y_test_pred_default = (y_test_proba >= 0.5).astype(int)
y_test_pred_tuned = (y_test_proba >= FINAL_THRESHOLD).astype(int)

# 4. In báo cáo chi tiết
print('=== KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST ===')

print(f'\n--- Tại ngưỡng mặc định (0.5) ---')
print(classification_report(y_test, y_test_pred_default, target_names=['Good', 'Bad'], zero_division=0))

print(f'--- Tại ngưỡng triển khai F1-optimal ({FINAL_THRESHOLD:.4f}) ---')
print(classification_report(y_test, y_test_pred_tuned, target_names=['Good', 'Bad'], zero_division=0))

print("-" * 50)
print(f'ROC-AUC (Test): {roc_auc_score(y_test, y_test_proba):.4f} (Đánh giá khả năng phân tách chung)')
print(f'PR-AUC  (Test): {average_precision_score(y_test, y_test_proba):.4f} (Đánh giá chuyên sâu trên tập mất cân bằng)')
print("-" * 50)

# 5. Hiển thị Ma trận nhầm lẫn (Confusion Matrix) cho ngưỡng tối ưu
cm_test = confusion_matrix(y_test, y_test_pred_tuned)
print(f'\nMa trận nhầm lẫn (Confusion Matrix) tại ngưỡng {FINAL_THRESHOLD:.4f}:')
display(pd.DataFrame(
    cm_test,
    index=['Thực tế: Good (0)', 'Thực tế: Bad (1)'],
    columns=['Dự đoán: Good (0)', 'Dự đoán: Bad (1)']
))

In [ ]:
# =============================================
# BIỂU ĐỒ 2: Confusion Matrix (ngưỡng 0.5 vs F1-optimal)
# =============================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_default = confusion_matrix(y_test, y_test_pred_default)
cm_test = confusion_matrix(y_test, y_test_pred_tuned)

ConfusionMatrixDisplay(cm_default, display_labels=['Good', 'Bad']).plot(ax=axes[0], cmap='Blues', values_format='d')
axes[0].set_title(f'Ngưỡng mặc định (0.5)\nF1={f1_score(y_test, y_test_pred_default):.4f}', fontsize=12)

ConfusionMatrixDisplay(cm_test, display_labels=['Good', 'Bad']).plot(ax=axes[1], cmap='Oranges', values_format='d')
axes[1].set_title(f'Ngưỡng F1-optimal ({FINAL_THRESHOLD:.4f})\nF1={f1_score(y_test, y_test_pred_tuned):.4f}', fontsize=12)

plt.suptitle('So sánh Confusion Matrix trên tập Test', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'confusion_matrix_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Đã lưu Confusion Matrix tại:', os.path.join(MODEL_DIR, 'confusion_matrix_comparison.png'))

In [ ]:
# =============================================
# BIỂU ĐỒ 3: Precision-Recall Curve trên Test
# =============================================
fig, ax = plt.subplots(figsize=(8, 6))
prec_test, rec_test, thr_test = precision_recall_curve(y_test, y_test_proba)
ax.plot(rec_test, prec_test, linewidth=2, color='#2196F3')
ax.axvline(x=recall_score(y_test, y_test_pred_tuned), color='red', linestyle='--', alpha=0.7,
           label=f'Ngưỡng F1-optimal ({FINAL_THRESHOLD:.4f})')
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title(f'Precision-Recall Curve trên Test\nPR-AUC = {average_precision_score(y_test, y_test_proba):.4f}',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'pr_curve_test.png'), dpi=150, bbox_inches='tight')
plt.show()

## 14. Kiểm chứng bất biến dữ liệu


Xác nhận rằng toàn bộ dữ liệu gốc (8 bảng CSV) và X_train_raw/X_test_raw **không bị thay đổi** qua toàn bộ quá trình cải tiến mô hình — đảm bảo nguyên tắc bất biến.

In [ ]:
all_ok = True
for name, df in raw.items():
    if df.shape != ORIG_SHAPES[name] or int(df.isnull().sum().sum()) != ORIG_NULL_COUNTS[name]:
        print(f'LỖI: bảng {name} đã bị thay đổi!')
        all_ok = False
if int(X_train_raw.isnull().sum().sum()) != TRAIN_NULL_BEFORE or int(X_test_raw.isnull().sum().sum()) != TEST_NULL_BEFORE:
    print('LỖI: X_train_raw/X_test_raw đã bị thay đổi!')
    all_ok = False
if all_ok:
    print('✔ Toàn bộ dữ liệu gốc và X_train_raw/X_test_raw KHÔNG bị thay đổi qua toàn bộ quá trình cải tiến mô hình.')

## 15. Đóng gói & lưu 3 file giao nộp


Lưu 3 file: `best_model.pkl` (pipeline đầy đủ), `preprocessing_pipeline.pkl` (chỉ phần ColumnTransformer), `model_metrics.json` (toàn bộ thông số, ngưỡng, metadata).

In [ ]:
def extract_preprocessor(estimator):
    """Trích xuất ColumnTransformer bên trong pipeline/CalibratedClassifierCV, để lưu riêng."""
    if hasattr(estimator, 'named_steps') and 'preprocessor' in estimator.named_steps:
        return estimator.named_steps['preprocessor']
    for attr_name in ('estimator', 'base_estimator'):
        inner = getattr(estimator, attr_name, None)
        if inner is not None:
            found = extract_preprocessor(inner)
            if found is not None:
                return found
    return None


best_model_name = best_pre_threshold_name
best_model = final_pipeline  # đã fit ở mục 13 trên X_train_raw

# 1) best_model.pkl — pipeline đầy đủ (preprocessor + classifier), fit sẵn
best_model_path = os.path.join(MODEL_DIR, 'best_model.pkl')
with open(best_model_path, 'wb') as f:
    pickle.dump(best_model, f)
print(f'✅ Đã lưu best_model.pkl ({best_model_name}) tại: {best_model_path}')

# 2) preprocessing_pipeline.pkl — chỉ phần ColumnTransformer
preprocessor_to_save = extract_preprocessor(best_model)
pipeline_path = os.path.join(MODEL_DIR, 'preprocessing_pipeline.pkl')
if preprocessor_to_save is not None:
    with open(pipeline_path, 'wb') as f:
        pickle.dump(preprocessor_to_save, f)
    print(f'✅ Đã lưu preprocessing_pipeline.pkl tại: {pipeline_path}')
else:
    print('⚠️  Không trích xuất được preprocessor riêng — kiểm tra lại cấu trúc best_model trước khi giao nộp.')

# 3) model_metrics.json
metrics_to_save = {}
for row in results_all:
    metrics_to_save[row['technique']] = {
        'pr_auc': round(row['pr_auc_mean'], 4),
        'roc_auc': round(row['roc_auc_mean'], 4),
        'accuracy': round(row['accuracy_mean'], 4),
        'recall': round(row['recall_mean'], 4),
        'precision': round(row['precision_mean'], 4),
        'f1': round(row['f1_mean'], 4),
    }

metrics_to_save['_meta'] = {
    'best_model': best_model_name,
    'selection_criteria': 'F1 Score cao nhất qua Stratified 5-Fold CV, sau đó tối ưu threshold',
    'task': 'Loan Status Classification (gộp theo hành vi trả nợ)',
    'label_encoding': {'0': 'Good (status A, C)', '1': 'Bad (status B, D)'},
    'decision_thresholds': {
        'default': 0.5,
        'f1_optimal_on_oof': round(float(threshold_f1_optimal), 4),   # giữ để tham khảo
        f'fbeta_{BETA}_optimal': round(float(threshold_fbeta_optimal), 4),
        'cost_optimal': round(float(threshold_cost_optimal), 4),
        'recommended_for_deployment': 0.5,                             # ← đổi thành 0.5
    },
    'feature_cols': numeric_features + categorical_features,
    'numeric_cols': numeric_features,
    'categorical_cols': categorical_features,
    'shap_selected_numeric_cols': best_shap_num,
    'shap_selected_categorical_cols': best_shap_cat,
    'test_size': 0.2,
    'random_state': RANDOM_STATE,
    'feature_engineering_notes': {
        'account_age_days': 'Số ngày từ khi mở tài khoản đến ngày cấp khoản vay',
        'n_disponents': 'Số người có quyền trên tài khoản (đếm theo disposition)',
        'is_local_client': '1 nếu chủ tài khoản cùng khu vực (district) với tài khoản',
        'has_card_before_loan': '1 nếu có thẻ được cấp TRƯỚC ngày vay (chống leakage)',
        'n_orders/total_order_amount/avg_order_amount': 'Thống kê lệnh thanh toán định kỳ của tài khoản',
        '*_before_loan (n_trans, avg/std/min/max balance, avg/total amount, last_balance)':
            'Thống kê giao dịch CHỈ tính trên các giao dịch có ngày TRƯỚC ngày vay (chống leakage thời gian)',
    },
    'data_sources': [
        'completedacct.csv', 'completedclient.csv', 'completeddisposition.csv',
        'completeddistrict.csv', 'completedloan.csv', 'completedcard.csv',
        'completedorder.csv', 'completedtrans.csv'
    ],
    'test_metrics_at_recommended_threshold': {
        'accuracy': round(float(accuracy_score(y_test, y_test_pred_default)), 4),
        'recall':   round(float(recall_score(y_test, y_test_pred_default)), 4),
        'precision':round(float(precision_score(y_test, y_test_pred_default)), 4),
        'f1':       round(float(f1_score(y_test, y_test_pred_default)), 4),
        'roc_auc':  round(float(roc_auc_score(y_test, y_test_proba)), 4),
        'pr_auc':   round(float(average_precision_score(y_test, y_test_proba)), 4),
        'confusion_matrix': confusion_matrix(y_test, y_test_pred_default).tolist(),
    },
}

metrics_path = os.path.join(MODEL_DIR, 'model_metrics.json')
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(metrics_to_save, f, indent=2, ensure_ascii=False)
print(f'✅ Đã lưu model_metrics.json tại: {metrics_path}')

print('\n=== ĐÃ HOÀN TẤT ĐÓNG GÓI KẾT QUẢ ===')
print(f'  1. {best_model_path}')
print(f'  2. {pipeline_path}')
print(f'  3. {metrics_path}')